# Murf Finch speech-to-text on Amazon SageMaker

This notebook deploys a Murf Finch model package from AWS Marketplace and shows how to call it:

1. Real-time transcription (`InvokeEndpoint`): send an audio file, get the transcript.
2. Language and word timestamps.
3. Batch transform: transcribe a folder of audio files from Amazon S3.

For live audio, use `InvokeEndpointWithBidirectionalStream`; see the README for the message format.

**Before you start**

- Subscribe to the listing in AWS Marketplace and pick a version: Finch Global (English and European languages) or Finch Indic (Indian languages).
- Make sure the account has quota for the instance type below (Service Quotas, "ml.g6.2xlarge for endpoint usage").
- Run this from SageMaker Studio or a notebook instance whose execution role has `AmazonSageMakerFullAccess`.

The container runs with network isolation: audio never leaves your account.

In [ ]:
import json, time, urllib.request
import boto3, sagemaker

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
sm = boto3.client("sagemaker", region_name=region)
runtime = boto3.client("sagemaker-runtime", region_name=region)

# Copy the model package ARN for your Region from the listing's "Configure" page.
MODEL_PACKAGE_ARN = "<model-package-arn-for-your-region>"
INSTANCE_TYPE = "ml.g6.2xlarge"   # also supported: ml.g6.xlarge, ml.g5.xlarge, ml.g5.2xlarge
name = f"murf-finch-{time.strftime('%Y%m%d-%H%M%S')}"

# Sample clips from the FLEURS dataset (CC BY 4.0).
SAMPLE_EN = urllib.request.urlopen("https://raw.githubusercontent.com/murf-ai/murf-finch-sagemaker/main/samples/en_us_sample.wav").read()
SAMPLE_HI = urllib.request.urlopen("https://raw.githubusercontent.com/murf-ai/murf-finch-sagemaker/main/samples/hi_in_sample.wav").read()

## Deploy a real-time endpoint

Creating the endpoint takes 10 to 20 minutes.

In [ ]:
sm.create_model(
    ModelName=name,
    ExecutionRoleArn=role,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
    EnableNetworkIsolation=True,
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": name,
        "InstanceType": INSTANCE_TYPE,
        "InitialInstanceCount": 1,
        "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
    }],
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
sm.get_waiter("endpoint_in_service").wait(EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 80})
print("endpoint", name, "is InService")

## 1. Real-time transcription

In [ ]:
t0 = time.perf_counter()
resp = runtime.invoke_endpoint(EndpointName=name, ContentType="audio/wav", Body=SAMPLE_EN)
result = json.loads(resp["Body"].read())
print(f"{result['audio_duration_secs']:.2f} s of audio in {time.perf_counter() - t0:.2f} s")
print(result["language_code"], ":", result["text"])

## 2. Language and word timestamps

Set options in `CustomAttributes`. Omit `language_code` to detect the language automatically. Word timestamps are returned by Finch Indic; Finch Global returns an empty `words` list.
With Finch Indic, use the Hindi sample and `language_code=hi`.

In [ ]:
resp = runtime.invoke_endpoint(EndpointName=name, ContentType="audio/wav", Body=SAMPLE_EN,
                               CustomAttributes="language_code=en,timestamps_granularity=word")
result = json.loads(resp["Body"].read())
for w in result["words"]:
    print(f"{w['start']:5.2f}-{w['end']:5.2f}  {w['text']}")

## 3. Batch transform

Each audio file in the input prefix is transcribed; the output prefix gets one JSON transcript per file.

In [ ]:
s3 = boto3.client("s3", region_name=region)
prefix = f"murf-finch-batch/{name}"
s3.put_object(Bucket=bucket, Key=f"{prefix}/input/en_us_sample.wav", Body=SAMPLE_EN)
s3.put_object(Bucket=bucket, Key=f"{prefix}/input/hi_in_sample.wav", Body=SAMPLE_HI)

sm.create_transform_job(
    TransformJobName=name,
    ModelName=name,
    MaxConcurrentTransforms=4,
    BatchStrategy="SingleRecord",
    MaxPayloadInMB=100,
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{bucket}/{prefix}/input/"}},
        "ContentType": "audio/wav",
        "SplitType": "None",
    },
    TransformOutput={"S3OutputPath": f"s3://{bucket}/{prefix}/output/", "Accept": "application/json"},
    TransformResources={"InstanceType": INSTANCE_TYPE, "InstanceCount": 1},
)
sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=name)
for obj in s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/output/").get("Contents", []):
    out = json.loads(s3.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read())
    print(obj["Key"].rsplit("/", 1)[-1], "->", out["text"])

## Clean up

The endpoint is billed while it runs. Delete it when you are done.

In [ ]:
sm.delete_endpoint(EndpointName=name)
sm.delete_endpoint_config(EndpointConfigName=name)
sm.delete_model(ModelName=name)